In [1]:
# BigQuery Pipeline Validation - Setup

# Import required libraries
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd

# --- CONFIG ---
PROJECT_ID = "linear-theater-436300-r9"
DATASET_ID = "ecommerce_pipeline"

# Initialize BigQuery client (uses your local gcloud auth or service account)
try:
    client = bigquery.Client(project=PROJECT_ID)
    print(f"Connected to BigQuery project: {PROJECT_ID}")
except Exception as e:
    print("Failed to connect to BigQuery:", e)

# List datasets to confirm access
datasets = list(client.list_datasets())
if datasets:
    print("Available datasets:")
    for ds in datasets:
        print(f" - {ds.dataset_id}")
else:
    print("No datasets found in this project.")


Connected to BigQuery project: linear-theater-436300-r9
Available datasets:
 - ecommerce_pipeline


In [3]:
# List tables and preview sample data from BigQuery
from IPython.display import display
# --- CONFIG ---
DATASET_ID = "ecommerce_pipeline"

# List all tables in the dataset
tables = list(client.list_tables(DATASET_ID))

if not tables:
    print(f"No tables found in dataset '{DATASET_ID}'.")
else:
    print(f"Tables in dataset '{DATASET_ID}':")
    for t in tables:
        print(f" - {t.table_id}")

    # Preview first few rows from each table
    for t in tables:
        print(f"\n=== Preview: {t.table_id} ===")
        query = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{t.table_id}` LIMIT 5"
        try:
            df_preview = client.query(query).to_dataframe()
            display(df_preview)
        except Exception as e:
            print(f"Error previewing {t.table_id}: {e}")


Tables in dataset 'ecommerce_pipeline':
 - amazon_pricing_ready
 - amazon_product_meta_cleaned
 - amazon_sentiment_cleaned

=== Preview: amazon_pricing_ready ===


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,price,average_rating,category,avg_sentiment,avg_helpful,review_count,log_price
0,guerra de capos malditos,3.42,1.0,All_Beauty,0.564764,0.977455,49812,1.486140
1,nueva crema hidratante para manos y pies asper...,16.98,1.0,All_Beauty,0.564764,0.977455,49812,2.889260
2,"n5 eau de parfum spray for women, 3.4 ounce100ml",99.00,1.0,All_Beauty,0.564764,0.977455,49812,4.605170
3,cinq mondes eau egyptienne fresh aromatic mist...,20.00,1.0,All_Beauty,0.564764,0.977455,49812,3.044522
4,"beeswax hair removal mousse, 2pcs 200ml mousse...",13.99,1.0,All_Beauty,0.564764,0.977455,49812,2.707383



=== Preview: amazon_product_meta_cleaned ===


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,title,price,average_rating,category
0,guerra de capos malditos,3.42,1.0,All_Beauty
1,nueva crema hidratante para manos y pies asper...,16.98,1.0,All_Beauty
2,"n5 eau de parfum spray for women, 3.4 ounce100ml",99.00,1.0,All_Beauty
3,cinq mondes eau egyptienne fresh aromatic mist...,20.00,1.0,All_Beauty
4,"beeswax hair removal mousse, 2pcs 200ml mousse...",13.99,1.0,All_Beauty



=== Preview: amazon_sentiment_cleaned ===


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,rating,text,title,asin,user_id,timestamp,verified_purchase,helpful_vote,category
0,1.0,felt synthetic,synthetic feeling,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,True,0,All_Beauty
1,1.0,nothing special unfortunately i waited too lon...,not what it claims to do,B01AKTGHFW,AHGAOIZVODNHYMNCBV4DECZH42UQ,1500768914981,True,0,All_Beauty
2,1.0,normally i like nyx products but not this one....,dont bother,B01M5KNSQN,AFETVW7S5M4LVJ7GTWPCKT7S3YBQ,1649634131604,True,0,All_Beauty
3,1.0,plastic does not bend for matching your brows ...,waste of money,B09FF97RHL,AGVVUU3QRQBHNASSGI5YQLPYOI2Q,1648824907536,True,0,All_Beauty
4,1.0,sizing is all off,weird sizing.,B087JNH3NC,AH4CGRSYSW5CWLRGQYRZKNJBUPAA,1598833004245,True,0,All_Beauty


In [5]:
# Pipeline Data Validation - Schema, Counts, and Sanity Checks

from google.cloud import bigquery
import pandas as pd

# --- CONFIG ---
DATASET_ID = "ecommerce_pipeline"
TABLES_TO_CHECK = ["amazon_sentiment_cleaned", "amazon_product_meta_cleaned", "amazon_pricing_ready"]

print(f"Validating tables in dataset: {DATASET_ID}\n")

for table_name in TABLES_TO_CHECK:
    full_table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    print(f"=== Checking: {table_name} ===")

    try:
        # Get table metadata
        table = client.get_table(full_table_id)
        schema = [(field.name, field.field_type) for field in table.schema]
        print(f"Schema: {schema}")

        # Count rows
        row_query = f"SELECT COUNT(*) as total_rows FROM `{full_table_id}`"
        total_rows = client.query(row_query).to_dataframe().iloc[0, 0]
        print(f"Row count: {total_rows}")

        # Compute null percentages
        null_query = f"""
        SELECT 
            {', '.join([f'ROUND(SUM(CASE WHEN {col.name} IS NULL THEN 1 ELSE 0 END)*100/COUNT(*),2) AS {col.name}_null_pct' for col in table.schema])}
        FROM `{full_table_id}`
        """
        null_df = client.query(null_query).to_dataframe()
        display(null_df)

        # Quick sanity check for numeric columns
        num_cols = [c.name for c in table.schema if c.field_type in ("INTEGER", "FLOAT", "NUMERIC")]
        if num_cols:
            stats_query = f"""
            SELECT 
                {', '.join([f'ROUND(AVG({c}),2) AS avg_{c}, MIN({c}) AS min_{c}, MAX({c}) AS max_{c}' for c in num_cols])}
            FROM `{full_table_id}`
            """
            stats_df = client.query(stats_query).to_dataframe()
            display(stats_df)

        print("\n")

    except Exception as e:
        print(f"Error validating {table_name}: {e}\n")


Validating tables in dataset: ecommerce_pipeline

=== Checking: amazon_sentiment_cleaned ===
Schema: [('rating', 'FLOAT'), ('text', 'STRING'), ('title', 'STRING'), ('asin', 'STRING'), ('user_id', 'STRING'), ('timestamp', 'INTEGER'), ('verified_purchase', 'BOOLEAN'), ('helpful_vote', 'INTEGER'), ('category', 'STRING')]


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Row count: 133730


,rating_null_pct,text_null_pct,title_null_pct,asin_null_pct,user_id_null_pct,timestamp_null_pct,verified_purchase_null_pct,helpful_vote_null_pct,category_null_pct
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


...

/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Row count: 90000


,title_null_pct,price_null_pct,average_rating_null_pct,category_null_pct,avg_sentiment_null_pct,avg_helpful_null_pct,review_count_null_pct,log_price_null_pct
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,avg_price,min_price,max_price,avg_average_rating,min_average_rating,max_average_rating,avg_avg_sentiment,min_avg_sentiment,max_avg_sentiment,avg_avg_helpful,min_avg_helpful,max_avg_helpful,avg_review_count,min_review_count,max_review_count,avg_log_price,min_log_price,max_log_price
0,28.44,0.0,12998.0,4.13,1.0,5.0,0.67,0.564764,0.767799,1.81,0.977455,2.981391,44576.67,34875,49812,2.84,0.0,9.472628


In [8]:
# === Cell 4: BigQuery Sanity Queries & Validation ===

from google.cloud import bigquery
import pandas as pd

client  # already initialized earlier

def q(sql):
    """Helper to run a query and return a DataFrame"""
    return client.query(sql).result().to_dataframe()

# Confirm tables present
tables = list(client.list_tables(f"{PROJECT_ID}.{DATASET_ID}"))
table_names = sorted([t.table_id for t in tables])
print("Tables found:", table_names)

def table_exists(name):
    return name in table_names

# --- 1) Sentiment/Review analytics ---
if table_exists("amazon_sentiment_cleaned"):
    print("\n=== Sentiment/Review Data Validation ===")
    fq = f"`{PROJECT_ID}.{DATASET_ID}.amazon_sentiment_cleaned`"

    # Basic stats by category
    print("Average rating, helpful votes, and verified purchase ratio by category:")
    display(q(f"""
        SELECT
          category,
          COUNT(*) AS total_reviews,
          ROUND(AVG(rating),2) AS avg_rating,
          ROUND(AVG(helpful_vote),2) AS avg_helpful_vote,
          ROUND(SUM(CASE WHEN verified_purchase THEN 1 ELSE 0 END) / COUNT(*), 2) AS verified_ratio
        FROM {fq}
        GROUP BY category
        ORDER BY avg_rating DESC
    """))

    # Sample reviews for inspection
    print("\nSample reviews:")
    display(q(f"""
        SELECT title, text, rating, helpful_vote, verified_purchase, category
        FROM {fq}
        ORDER BY RAND()
        LIMIT 5
    """))


# --- 2) Product meta analytics ---
for meta_tbl in ["amazon_product_meta_cleaned", "amazon_product_meta"]:
    if table_exists(meta_tbl):
        print(f"\n=== Product Meta Data Validation ({meta_tbl}) ===")
        fq = f"`{PROJECT_ID}.{DATASET_ID}.{meta_tbl}`"
        display(q(f"""
            SELECT
              category,
              COUNT(*) AS n,
              ROUND(AVG(price),2) AS avg_price,
              ROUND(STDDEV(price),2) AS price_std,
              MIN(price) AS min_price,
              MAX(price) AS max_price,
              ROUND(AVG(average_rating),2) AS avg_rating
            FROM {fq}
            GROUP BY category
            ORDER BY avg_price DESC
        """))
        break


# --- 3) Pricing analytics ---
for pricing_tbl in ["amazon_pricing_ready", "amazon_pricing_analytics"]:
    if table_exists(pricing_tbl):
        print(f"\n=== Pricing Data Validation ({pricing_tbl}) ===")
        fq = f"`{PROJECT_ID}.{DATASET_ID}.{pricing_tbl}`"
        display(q(f"""
            SELECT
              category,
              COUNT(*) AS n,
              ROUND(AVG(price),2) AS avg_price,
              ROUND(AVG(average_rating),2) AS avg_product_rating,
              ROUND(AVG(avg_sentiment),2) AS avg_sentiment,
              ROUND(AVG(avg_helpful),2) AS avg_helpful,
              ROUND(AVG(review_count),2) AS avg_review_count
            FROM {fq}
            GROUP BY category
            ORDER BY avg_price DESC
        """))
        break

print("\nBigQuery pipeline validation — all tables accessible and consistent with expected schema.")


Tables found: ['amazon_pricing_ready', 'amazon_product_meta_cleaned', 'amazon_sentiment_cleaned']

=== Sentiment/Review Data Validation ===
Average rating, helpful votes, and verified purchase ratio by category:


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category,total_reviews,avg_rating,avg_helpful_vote,verified_ratio
0,Books,34875,4.35,2.98,0.51
1,Electronics,49043,4.24,1.46,0.78
2,All_Beauty,49812,4.05,0.98,0.85



Sample reviews:


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


...

,category,n,avg_price,price_std,min_price,max_price,avg_rating
0,Electronics,30000,46.94,203.17,0.39,12998.00,4.08
1,Books,30000,21.27,48.75,0.00,3226.05,4.40
2,All_Beauty,30000,17.13,20.60,0.12,2143.46,3.92



=== Pricing Data Validation (amazon_pricing_ready) ===


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category,n,avg_price,avg_product_rating,avg_sentiment,avg_helpful,avg_review_count
0,Electronics,30000,46.94,4.08,0.67,1.46,49043.0
1,Books,30000,21.27,4.40,0.77,2.98,34875.0
2,All_Beauty,30000,17.13,3.92,0.56,0.98,49812.0



BigQuery pipeline validation — all tables accessible and consistent with expected schema.
